# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order**.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row represents exactly **one page** (content piece) at the time of the snapshot.

**Time Window:** This analysis uses the `content_refresh_anonymized.csv` snapshot (representing a recent cross-section rather than a partitioned warehouse month like 2026-03). The metrics within it cover a trailing 90-day window (e.g., `impressions_90d`).

**Target/Label:** We predict `trend_direction == 'down'` as a binary classification target.

In [1]:
import pandas as pd
import os

data_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path)
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
print(f"Loaded {df.shape[0]} rows from the starter dataset.")

Loaded 30000 rows from the starter dataset.


## 2. Fields: feature / label / context / excluded

- **Context (Keys):** `content_id`, `client_id`.
- **Features:** `content_age_days`, `impressions_90d`, `avg_position`, `ctr`, `days_since_last_update`. These are observable before any intervention.
- **Label (Target Proxy):** `is_declining_label` (derived from `trend_direction`).
- **Excluded:** `trend_pct` and `trend_direction` (excluded from features to prevent leakage because they directly determine the target label).

In [2]:
context_cols = ['content_id', 'client_id']
feature_cols = ['content_age_days', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update']
label_col = 'is_declining_label'
excluded_cols = ['trend_pct', 'trend_direction']

print("Fields categorized successfully.")

Fields categorized successfully.


## 3. Verify it with queries (grain, counts, missing values, windows)

Let's prove the grain is truly one row per page, check the overall row counts, and ensure our availability criteria (`is_active` or minimum impressions) works.

In [3]:
# 1. Check grain: One row = one content_id
is_unique = df['content_id'].nunique() == len(df)
print(f"Grain check passed (1 row = 1 content_id): {is_unique}")

# 2. Counts and basic filter (Availability)
# Assuming we only care about pages with >0 impressions in the last 90 days
df_filtered = df[df['impressions_90d'] > 0].copy()
print(f"Rows surviving minimum visibility filter (>0 impressions): {len(df_filtered)} out of {len(df)}")

# 3. Building the 5 features
features_df = df_filtered[context_cols + feature_cols + [label_col]].copy()
print("\nFeature Frame Sample (all knowable at decision time):")
display(features_df.head(3))

# 4. THE TRAP (Leakage Experiment)
print("\n--- THE LEAKAGE TRAP ---")
features_df_leaky = df_filtered[feature_cols + ['trend_pct', label_col]].dropna()
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

X_clean = features_df.dropna()[feature_cols]
y_clean = features_df.dropna()[label_col]
model_clean = DecisionTreeClassifier(max_depth=3).fit(X_clean, y_clean)
print(f"Honest Model Score (Clean): {roc_auc_score(y_clean, model_clean.predict_proba(X_clean)[:, 1]):.3f}")

X_leaky = features_df_leaky.drop(columns=[label_col])
y_leaky = features_df_leaky[label_col]
model_leaky = DecisionTreeClassifier(max_depth=3).fit(X_leaky, y_leaky)
print(f"Trap Model Score (Leaky): {roc_auc_score(y_leaky, model_leaky.predict_proba(X_leaky)[:, 1]):.3f}")
print("The leaky model uses 'trend_pct', which mathematically dictates the label, causing near-perfect (fraudulent) scores.")

Grain check passed (1 row = 1 content_id): True
Rows surviving minimum visibility filter (>0 impressions): 30000 out of 30000

Feature Frame Sample (all knowable at decision time):


,content_id,client_id,content_age_days,impressions_90d,avg_position,ctr,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,187,3803,10.6,0.76,20,1
1,content_a1fb4e703a9e,client_4e07408562,445,15320,20.3,0.05,25,1
2,content_9aa793d4d895,client_7f2253d7e2,141,12581,36.5,0.09,20,1



--- THE LEAKAGE TRAP ---


Honest Model Score (Clean): 0.677
Trap Model Score (Leaky): 1.000
The leaky model uses 'trend_pct', which mathematically dictates the label, causing near-perfect (fraudulent) scores.


## 4. Data limits

- **No historical partitioning:** Using the starter dataset means we only have a single snapshot of time (the trailing 90 days). We cannot easily analyze how a page performed a year ago compared to two years ago.
- **Survivor bias:** Pages that were deleted entirely before this snapshot are not visible to the model.
- **Lack of qualitative content:** We have meta-metrics (like `word_count` and `content_age_days`) but no actual text content, meaning we cannot build semantic features (like whether the article contains outdated facts).

In [4]:
print("Data limits acknowledged.")

Data limits acknowledged.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.